# Notebook 02 -- Data Ingestion & Tag Mapping

## Objective

Produce one clean, merged, typed table -- process tags + crude assay properties, on a daily index, with TAM/shutdown
periods removed -- that every later notebook in this pipeline reads instead of touching the raw Excel again.
Concretely:

1. Parse the raw historian export into a typed daily DataFrame (reusing Notebook 01's validated tag inventory rather
   than re-deriving it).
2. Remove TAM/shutdown windows using the plant's own recovery signal (crude flow, CIT, COT), not a fixed calendar
   date range -- ported verbatim from the legacy pipeline's validated logic.
3. Merge in crude assay properties (API, SG, viscosity, MCRT, asphaltenes), aligned onto the daily process index.

This notebook does **not** correct sensor outliers, check chain consistency, or compute a data-quality score -- that
is Notebook 03's job (see docs/04's mapping). This notebook's job is *getting a clean, complete table*, not
*validating whether individual readings are physically plausible*.

## Inputs

| Input | Path | Note |
|---|---|---|
| Raw process Excel | `src.cpht.config.RAW_EXCEL` | Same file Notebook 01 already validated |
| Notebook 01's data profile | `Data/v2/data_profile.json` | Cross-check: this run should see the same tag count/date range Notebook 01 saw |
| Notebook 01's data dictionary | `Data/v2/data_dictionary.csv` | Tag list source -- avoids re-deriving the header inventory |
| Crude assay | `src.cpht.config.CRUDE_PROPERTY_CSV` (`Crude_property_profiled.csv`) | Built by `notebooks/00_data_prep_crude_assay.ipynb`; **static**, not re-run by this pipeline (see docs/ANALYSIS_PIPELINE_GUIDE.md item 3) |

## Assumptions

- Notebook 01 has already been run (this notebook fails fast in Data Validation if its outputs are missing) and its
  structural diagnostics passed.
- TAM/shutdown detection thresholds (`SHUTDOWN_FLOW_THRESHOLD=200`, `RECOVERY_FLOW_THRESHOLD=400`,
  `SHUTDOWN_MARGIN_DAYS=7` m3/hr and days) are **ported verbatim from the legacy pipeline**, not re-derived here --
  these are already validated against real TAM history (see `docs/04_Notebook_Pipeline_Redesign.md`'s "do not
  re-litigate" list). Re-deriving them is explicitly out of scope for this notebook.
- Crude assay is assumed constant between lab samples (forward/backward-filled onto the daily index) -- matches the
  legacy pipeline's approach; no batch/effective-date tracking exists to do better than this.

## Requirements traced from `docs/03`

| Requirement | Source | How this notebook addresses it |
|---|---|---|
| FR-DI-002 -- import Crude Assay (Excel/CSV/DB), check schema and effective date | docs/03 section 2.3 | Crude CSV read + column check in Data Validation; effective-date tracking is a known gap (see Limitations) |
| FR-DI-006 -- must not modify source data, keep raw immutable ("Bronze layer") | docs/03 section 2.2 | Raw Excel is only ever read, never written; this notebook's output is a new derived artifact, not an edit in place |
| IS-005 -- Crude Assay: API, SG, Viscosity and other thermophysical properties | docs/03 section 1.4 | Merged columns are exactly this set (see Method) |


## Method

1. Load `data_profile.json` + `data_dictionary.csv` from Notebook 01; fail fast if either is missing (that's the
   "validate at the boundary" behavior -- don't let this notebook silently re-derive a stale tag list).
2. Re-read the raw Excel body (same row/column offsets as Notebook 01) into a typed daily DataFrame, using the tag
   list from `data_dictionary.csv` for column naming.
3. Detect shutdown days (`total_crude_flow < SHUTDOWN_FLOW_THRESHOLD`), then expand each shutdown group backward and
   forward until flow/CIT/COT all recover to normal levels, then pad `SHUTDOWN_MARGIN_DAYS` on each side. Drop those
   rows.
4. Read the crude assay CSV, reindex/forward-fill/back-fill it onto the (post-shutdown-removal) process index, and
   join.
5. Write the merged table as this notebook's output artifact for Notebook 03.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks_v2" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import json

from src.cpht import config
from src.cpht import validation

print(f"Raw Excel:         {config.RAW_EXCEL}")
print(f"Crude assay CSV:   {config.CRUDE_PROPERTY_CSV}")
print(f"V2 output dir:     {config.V2_OUTPUT_DIR}")


Raw Excel:         C:\Desktop\Bangchak Internship 2026\Data\Process information data (2024-2026).xlsx
Crude assay CSV:   C:\Desktop\Bangchak Internship 2026\Data\Crude_property_profiled.csv
V2 output dir:     C:\Desktop\Bangchak Internship 2026\Data\v2


## Data Validation

Check Notebook 01's outputs exist and are internally consistent before doing any ingestion work. If Notebook 01
hasn't been run (or was run against a different raw file), stop here rather than silently re-deriving a tag list
that might not match what was actually validated.


In [2]:
v2_dir = Path(config.V2_OUTPUT_DIR)
profile_path = v2_dir / "data_profile.json"
dict_path = v2_dir / "data_dictionary.csv"

nb01_result = validation.ValidationResult(ok=True)
if not profile_path.exists():
    nb01_result.ok = False
    nb01_result.errors.append(f"{profile_path} not found -- run Notebook 01 first")
if not dict_path.exists():
    nb01_result.ok = False
    nb01_result.errors.append(f"{dict_path} not found -- run Notebook 01 first")

print(nb01_result.report())
nb01_result.raise_if_failed()

with open(profile_path, encoding="utf-8") as f:
    nb01_profile = json.load(f)
data_dictionary = pd.read_csv(dict_path)

print(f"\nNotebook 01 profile: {nb01_profile['n_tags_found']} tags, "
      f"{nb01_profile['n_rows']} rows, {nb01_profile['date_min']}..{nb01_profile['date_max']}")
print(f"structural_diagnostics_pass: {nb01_profile['structural_diagnostics_pass']}")
assert nb01_profile["structural_diagnostics_pass"], "Notebook 01's diagnostics did not pass -- fix that first"
assert nb01_profile["required_tags_present"], "Notebook 01 found missing required tags -- fix that first"

tag_list = data_dictionary["tag_id"].tolist()
print(f"Tag list loaded from Notebook 01's data dictionary: {len(tag_list)} tags")


Validation: PASS

Notebook 01 profile: 97 tags, 2008 rows, 2021-01-01T00:00:00..2026-07-01T00:00:00
structural_diagnostics_pass: True
Tag list loaded from Notebook 01's data dictionary: 97 tags


## Analysis

### 1. Parse the raw data body


In [3]:
# Same row/column offsets as Notebook 01 and the legacy notebooks/01_data_cleaning.ipynb --
# a structural fact about the raw file, not a design choice to redo.
raw = pd.read_excel(config.RAW_EXCEL, sheet_name="Sheet1", header=None)

data = raw.iloc[7:, 2:].copy()
data.columns = ["Timestamp"] + tag_list
data["Timestamp"] = pd.to_datetime(data["Timestamp"], errors="coerce")
data = data.dropna(subset=["Timestamp"]).set_index("Timestamp")

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print(f"Parsed: {len(data):,} rows, {len(data.columns)} tag columns")
assert len(data) == nb01_profile["n_rows"], (
    f"Row count mismatch vs Notebook 01's profile ({len(data)} vs {nb01_profile['n_rows']}) -- "
    "the raw file may have changed since Notebook 01 ran; re-run Notebook 01 first."
)


Parsed: 2,008 rows, 97 tag columns


### 2. Detect and remove TAM/shutdown windows

Ported verbatim from `notebooks/01_data_cleaning.ipynb`: flag low-flow days, then walk each shutdown group backward
and forward until crude flow, CIT, and COT all recover to normal levels, then pad `SHUTDOWN_MARGIN_DAYS` on each
side before dropping.


In [4]:
COT_TAG = "1TC007.pv"  # Coil Outlet Temp -- used as a shutdown-recovery signal, not itself part of HX_CONFIG

for required_col in (config.TOTAL_CHARGE_TAG, config.CIT_TAG, COT_TAG):
    assert required_col in data.columns, f"'{required_col}' missing from parsed columns -- cannot detect TAM windows"

data["total_crude_flow"] = data[config.TOTAL_CHARGE_TAG]
data["is_shutdown"] = data["total_crude_flow"] < config.SHUTDOWN_FLOW_THRESHOLD
n_flagged_initial = int(data["is_shutdown"].sum())
print(f"Initially flagged low-flow days: {n_flagged_initial}")

flow = data["total_crude_flow"]
cit = data[config.CIT_TAG]
cot = data[COT_TAG]
normal_cit = data.loc[~data["is_shutdown"], config.CIT_TAG].quantile(0.05)
normal_cot = data.loc[~data["is_shutdown"], COT_TAG].quantile(0.05)

shutdown_mask = data["is_shutdown"].copy()
groups = (shutdown_mask != shutdown_mask.shift()).cumsum()
for group_id in data[shutdown_mask].groupby(groups[shutdown_mask]).groups:
    group_idx = data[shutdown_mask].groupby(groups[shutdown_mask]).get_group(group_id).index
    start_pos = data.index.get_loc(group_idx.min())
    end_pos = data.index.get_loc(group_idx.max())

    while start_pos > 0:
        prev = start_pos - 1
        flow_ok = pd.notna(flow.iloc[prev]) and flow.iloc[prev] >= config.RECOVERY_FLOW_THRESHOLD
        cit_ok = pd.notna(cit.iloc[prev]) and cit.iloc[prev] >= normal_cit
        cot_ok = pd.notna(cot.iloc[prev]) and cot.iloc[prev] >= normal_cot
        if flow_ok and cit_ok and cot_ok:
            break
        start_pos -= 1

    while end_pos < len(data) - 1:
        nxt = end_pos + 1
        flow_ok = pd.notna(flow.iloc[nxt]) and flow.iloc[nxt] >= config.RECOVERY_FLOW_THRESHOLD
        cit_ok = pd.notna(cit.iloc[nxt]) and cit.iloc[nxt] >= normal_cit
        cot_ok = pd.notna(cot.iloc[nxt]) and cot.iloc[nxt] >= normal_cot
        if flow_ok and cit_ok and cot_ok:
            break
        end_pos += 1

    start_pos = max(0, start_pos - config.SHUTDOWN_MARGIN_DAYS)
    end_pos = min(len(data) - 1, end_pos + config.SHUTDOWN_MARGIN_DAYS)
    data.iloc[start_pos:end_pos + 1, data.columns.get_loc("is_shutdown")] = True

n_flagged_expanded = int(data["is_shutdown"].sum())
print(f"After backward/forward expansion + {config.SHUTDOWN_MARGIN_DAYS}-day margin: {n_flagged_expanded} days flagged")

data_cleaned = data[~data["is_shutdown"]].copy()
# Drop the internal working columns -- they did their job (detecting the shutdown
# windows) and shouldn't leak into the output as if they were real process tags.
data_cleaned = data_cleaned.drop(columns=["total_crude_flow", "is_shutdown"])
print(f"Rows after TAM/shutdown removal: {len(data_cleaned):,} (removed {len(data) - len(data_cleaned)})")
print(f"Columns after dropping internal working columns: {len(data_cleaned.columns)} (should equal {len(tag_list)})")
assert len(data_cleaned.columns) == len(tag_list)


Initially flagged low-flow days: 54
After backward/forward expansion + 7-day margin: 110 days flagged
Rows after TAM/shutdown removal: 1,898 (removed 110)
Columns after dropping internal working columns: 97 (should equal 97)


### 3. Merge crude assay properties

Crude grade is assumed constant between lab samples -- reindex the assay onto the (post-removal) process index and
forward/back-fill.


In [5]:
crude_prop = pd.read_csv(config.CRUDE_PROPERTY_CSV, index_col="Date", parse_dates=True).sort_index()
print(f"Crude assay: {len(crude_prop)} records, {crude_prop.index.min().date()}..{crude_prop.index.max().date()}")
print(f"Columns: {list(crude_prop.columns)}")

crude_prop_aligned = crude_prop.reindex(data_cleaned.index).ffill().bfill()
n_unfilled = crude_prop_aligned.isna().any(axis=1).sum()
if n_unfilled:
    print(f"WARNING: {n_unfilled} row(s) still have no crude property after ffill/bfill "
          f"(process date range extends beyond assay coverage)")

data_with_crude = data_cleaned.join(crude_prop_aligned)
print(f"\nMerged table: {data_with_crude.shape[0]:,} rows x {data_with_crude.shape[1]} columns")


Crude assay: 1869 records, 2021-04-01..2026-06-25
Columns: ['API', 'SG_15_6C', 'Visc_50C_cSt', 'Visc_100C_cSt', 'MCRT_pct', 'Asphaltenes_pct']

Merged table: 1,898 rows x 103 columns


## Diagnostic Checks

Cross-check against the legacy pipeline's known output (`Process_information_with_crude.csv`, as documented in
`docs/04_Notebook_Pipeline_Redesign.md`): same TAM-removal logic on the same raw file should produce a comparable
row count and no unexpected new nulls in required columns.


In [6]:
diag = validation.ValidationResult(ok=True)

# 1. Timestamp integrity on the final merged table.
ts_check = validation.check_timestamp_column(data_with_crude.reset_index(), timestamp_col="Timestamp")
diag = validation.combine(diag, ts_check)

# 2. Expected crude columns present (docs/03 IS-005).
expected_crude_cols = {"API", "SG_15_6C", "Visc_50C_cSt", "Visc_100C_cSt", "MCRT_pct", "Asphaltenes_pct"}
missing_crude_cols = expected_crude_cols - set(data_with_crude.columns)
if missing_crude_cols:
    diag.ok = False
    diag.errors.append(f"expected crude columns missing: {missing_crude_cols}")

# 3. Required-tag missing % should not have gotten WORSE than what Notebook 01 measured on the raw file
#    (TAM removal should only remove known-bad rows, not introduce new gaps in good ones).
required_tags = [t for t in config.required_tags() if t in data_with_crude.columns]
post_missing_pct = (data_with_crude[required_tags].isna().mean() * 100).max()
if post_missing_pct > nb01_profile["worst_required_tag_missing_pct"] + 1.0:  # 1pp tolerance for rounding
    diag.warnings.append(
        f"worst required-tag missing % after cleaning ({post_missing_pct:.2f}%) is higher than "
        f"Notebook 01's raw-file measurement ({nb01_profile['worst_required_tag_missing_pct']:.2f}%) -- unexpected"
    )

# 4. Sanity comparison against the legacy pipeline's last-known output shape (informational, not a hard
#    assertion -- the raw file has grown since that snapshot, so an exact match isn't expected; a wildly
#    different removal rate would be worth investigating).
LEGACY_KNOWN_ROWS_AFTER_TAM_REMOVAL = 1898  # Process_information_cleaned.csv, per docs/04
removal_rate = (len(data) - len(data_cleaned)) / len(data)
print(f"This run removed {removal_rate:.1%} of rows as TAM/shutdown "
      f"({len(data):,} -> {len(data_cleaned):,}); "
      f"legacy pipeline's last snapshot had {LEGACY_KNOWN_ROWS_AFTER_TAM_REMOVAL:,} rows after the same step.")

print(diag.report())
diag.raise_if_failed()


This run removed 5.5% of rows as TAM/shutdown (2,008 -> 1,898); legacy pipeline's last snapshot had 1,898 rows after the same step.
Validation: PASS


## Results


In [7]:
print(f"Raw rows parsed:          {len(data):,}")
print(f"TAM/shutdown days removed: {len(data) - len(data_cleaned):,} ({removal_rate:.1%})")
print(f"Rows after cleaning:      {len(data_cleaned):,}")
print(f"Crude columns merged:     {sorted(expected_crude_cols)}")
print(f"Final table shape:        {data_with_crude.shape[0]:,} rows x {data_with_crude.shape[1]} columns")
print(f"Diagnostics:              {'PASS' if diag.ok else 'FAIL'} ({len(diag.warnings)} warning(s))")


Raw rows parsed:          2,008
TAM/shutdown days removed: 110 (5.5%)
Rows after cleaning:      1,898
Crude columns merged:     ['API', 'Asphaltenes_pct', 'MCRT_pct', 'SG_15_6C', 'Visc_100C_cSt', 'Visc_50C_cSt']
Final table shape:        1,898 rows x 103 columns
Diagnostics:              PASS (0 warning(s))


## Limitations

- TAM/shutdown detection thresholds are ported verbatim from the legacy pipeline (see Assumptions) -- not re-derived
  or re-validated here. If the plant's operating envelope shifts significantly (e.g. a new normal minimum flow),
  these thresholds would need re-tuning, which is out of scope for this notebook.
- Reduced-rate operation (partial throughput, not a full shutdown) is deliberately **not** removed -- only windows
  matching the shutdown+recovery pattern are dropped. A sustained partial-rate period would remain in the data and
  should be caught by Notebook 03's data-quality checks, not this one.
- Crude assay merge assumes constant grade between lab samples (forward/backward-fill) -- there is no batch or
  effective-date tracking to do better than this (matches docs/03's FR-DI-002 gap: "effective date" checking is not
  implemented, just date-based ffill/bfill).
- This notebook does **not** correct sensor outliers or check chain consistency (e.g. `cold_out < cold_in`) --
  Notebook 03 owns that. A row can pass this notebook and still contain a physically implausible reading.

## Outputs

Writes `process_with_crude.csv` (typed, TAM-removed, crude-merged) to `src.cpht.config.V2_OUTPUT_DIR` -- Notebook 03
(Data Quality) reads this rather than re-parsing the raw Excel.


In [8]:
out_dir = Path(config.V2_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "process_with_crude.csv"
data_with_crude.to_csv(out_path, encoding="utf-8")
print(f"Wrote {out_path}  ({data_with_crude.shape[0]:,} rows x {data_with_crude.shape[1]} columns)")

ingestion_summary = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "rows_parsed": int(len(data)),
    "rows_after_tam_removal": int(len(data_cleaned)),
    "removal_rate": float(removal_rate),
    "final_shape": list(data_with_crude.shape),
    "crude_columns": sorted(expected_crude_cols),
    "diagnostics_pass": bool(diag.ok),
    "diagnostic_warnings": diag.warnings,
}
summary_path = out_dir / "ingestion_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(ingestion_summary, f, indent=2, default=str)
print(f"Wrote {summary_path}")


Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\process_with_crude.csv  (1,898 rows x 103 columns)
Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\ingestion_summary.json


## Conclusion

The raw historian export was parsed, TAM/shutdown windows were removed using the legacy pipeline's validated
recovery-signal logic, and crude assay properties were merged onto the daily process index. The result is one clean,
typed, merged table ready for outlier/quality checks -- this notebook does not itself judge whether individual
readings are physically plausible (see Limitations).

## Next Notebook

**Notebook 03 -- Data Quality**: reads `process_with_crude.csv` from this notebook, applies outlier/chain-consistency
correction (ported from `notebooks/01_data_cleaning.ipynb`'s second half), and -- unlike the legacy pipeline --
builds the combined per-HX Data Quality Score required by `docs/03` (FR-DQ-012), reading `data_dictionary.csv` from
Notebook 01 rather than re-deriving tag metadata.
